# Colab 04 — ¿De qué material está hecho este cuerpo?

**Laboratorio 1 · Departamento de Física · FCEN-UBA**

Clase 4 — 02/09

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/charlyacha/Labo1-colabs/blob/main/04_Propagacion_de_incertezas.ipynb)

Mediste masa y dimensiones. Querés la densidad, que no se mide: se calcula. La pregunta es con qué incerteza, porque de eso depende que puedas decir si es aluminio o es una aleación.

**Al terminar vas a poder:** propagar incertezas con la fórmula general, identificar qué medición domina el error, y reconocer un error sistemático que el promedio no arregla.

---

### Antes de tocar nada

Andá a **Archivo → Guardar una copia en Drive**. Vas a trabajar sobre tu copia:
lo que escribas acá sin copiar primero no se guarda en ningún lado.

Este cuaderno se recorre **de arriba hacia abajo**. Las celdas no son
independientes: cada una usa lo que definieron las anteriores. Si algo tira
`NameError`, casi siempre es porque salteaste una celda.

In [ ]:
import os

if not os.path.exists("lab1_utils.py"):
    !wget -q -O lab1_utils.py https://raw.githubusercontent.com/charlyacha/Labo1-colabs/main/lab1_utils.py

import numpy as np
import matplotlib.pyplot as plt
import lab1_utils as lab

lab.estilo_lab1()
print("Listo. numpy", np.__version__)

### 1. El problema

Un cilindro macizo: masa $m$, diámetro $D$, altura $h$. La densidad es

$$\rho = \frac{m}{V} = \frac{4m}{\pi D^2 h}$$

Cada una de las tres mediciones tiene su incerteza. ¿Cuál es la incerteza de
$\rho$?

La respuesta general, para $f(x_1, \dots, x_n)$ con variables
**independientes**:

$$\sigma_f^2 = \sum_i \left(\frac{\partial f}{\partial x_i}\right)^2 \sigma_{x_i}^2$$

Dos advertencias sobre esa fórmula, que casi nunca se enuncian:

- Es una **aproximación de primer orden**. Vale mientras las incertezas sean
  chicas frente a los valores, que es el caso normal en el laboratorio.
- Supone variables **independientes**. Si medís $D$ y $h$ con el mismo
  calibre mal calibrado, los errores están correlacionados y hay un término
  cruzado que esta fórmula ignora.

In [ ]:
# Mediciones de ejemplo. Reemplazá por las tuyas.
m, sm = 29.44, 0.01          # gramos
D, sD = 21.46, 0.05          # milímetros
h, sh = 30.15, 0.05          # milímetros

print(f"masa    : {lab.formatear(m, sm, 'g')}")
print(f"diámetro: {lab.formatear(D, sD, 'mm')}")
print(f"altura  : {lab.formatear(h, sh, 'mm')}")

### 2. Derivar sin equivocarse: sympy

Derivar a mano tres veces y no confundirse un signo es posible, pero es
tiempo gastado en algo que la máquina hace exacto. `sympy` deriva de forma
**simbólica**: te devuelve la fórmula, no un número.

In [ ]:
import sympy as sp

m_s, D_s, h_s = sp.symbols("m D h", positive=True)

rho_s = 4 * m_s / (sp.pi * D_s**2 * h_s)

print("rho =")
sp.pprint(rho_s)
print()
print("derivadas parciales:")
for var in (m_s, D_s, h_s):
    print(f"  d(rho)/d({var}) =", sp.simplify(sp.diff(rho_s, var)))

Ahora armamos la fórmula de propagación completa, también simbólicamente, y
recién al final la evaluamos con `lambdify`, que convierte una expresión de
sympy en una función de numpy.

In [ ]:
sm_s, sD_s, sh_s = sp.symbols("sigma_m sigma_D sigma_h", positive=True)

variables = [(m_s, sm_s), (D_s, sD_s), (h_s, sh_s)]

# Cada término de la suma en cuadratura.
terminos = [(sp.diff(rho_s, v) * s)**2 for v, s in variables]
sigma_rho_s = sp.sqrt(sum(terminos))

f_rho = sp.lambdify((m_s, D_s, h_s), rho_s, "numpy")
f_sigma = sp.lambdify((m_s, D_s, h_s, sm_s, sD_s, sh_s), sigma_rho_s, "numpy")

rho = f_rho(m, D, h)
sigma_rho = f_sigma(m, D, h, sm, sD, sh)

# Pasamos de g/mm³ a g/cm³ multiplicando por 1000.
lab.reportar(rho * 1000, sigma_rho * 1000, "g/cm³", nombre="densidad")

### 3. Lo que realmente sirve: la tabla de contribuciones

Propagar no es un trámite para poner un número después del ±. Sirve para
contestar **cuál de todas tus mediciones conviene mejorar**, que es una
pregunta de diseño experimental con impacto en tu tiempo.

In [ ]:
nombres = ["masa", "diámetro", "altura"]
valores = [m, D, h]
sigmas = [sm, sD, sh]

contribuciones = []
for (v_sym, s_sym), val, sig in zip(variables, valores, sigmas):
    derivada = sp.lambdify((m_s, D_s, h_s), sp.diff(rho_s, v_sym), "numpy")
    contribuciones.append((derivada(m, D, h) * sig)**2)

contribuciones = np.array(contribuciones)
porcentaje = 100 * contribuciones / contribuciones.sum()

print("variable      contribución a la varianza de rho")
for n, p in zip(nombres, porcentaje):
    barra = "#" * int(round(p / 2))
    print(f"{n:<12} {p:5.1f} %  {barra}")

El diámetro domina, y no porque esté peor medido: entra **al cuadrado**, así
que su error relativo pesa el doble. Mejorar la balanza no te sirve de nada;
medir el diámetro en cinco posiciones distintas y promediar, sí.

**Regla general:** en un producto de potencias
$f = x^a y^b z^c$, los errores **relativos** se suman en cuadratura pesados
por los exponentes:

$$\left(\frac{\sigma_f}{f}\right)^2 = a^2\left(\frac{\sigma_x}{x}\right)^2
+ b^2\left(\frac{\sigma_y}{y}\right)^2 + c^2\left(\frac{\sigma_z}{z}\right)^2$$

In [ ]:
# Verificación de la regla con este caso: rho = 4 m D^-2 h^-1
rel = np.sqrt((sm/m)**2 + (2*sD/D)**2 + (sh/h)**2)

print(f"error relativo por la regla de potencias: {100*rel:.4f} %")
print(f"error relativo por sympy               : {100*sigma_rho/rho:.4f} %")

### 4. Contestar la pregunta del título

In [ ]:
tabla = {"aluminio": 2.70, "titanio": 4.51, "bronce": 8.80,
         "cobre": 8.96, "acero": 7.85}

rho_cgs, srho_cgs = rho * 1000, sigma_rho * 1000

print(f"Tu resultado: {lab.formatear(rho_cgs, srho_cgs, 'g/cm³')}")
print()
for material, valor in tabla.items():
    z = abs(rho_cgs - valor) / srho_cgs
    veredicto = "COMPATIBLE" if z < 3 else "descartado"
    print(f"  {material:<10} {valor:5.2f}   z = {z:7.1f}   {veredicto}")

Con estos números queda un solo candidato en pie, y eso **es** la respuesta a
la pregunta del título. Pero fijate lo frágil que es esa conclusión: alcanza
con haber medido el diámetro con regla en lugar de calibre para que la
incerteza se multiplique por veinte y sobrevivan dos o tres materiales. Ahí la
respuesta honesta pasa a ser "es uno de estos dos", y la tabla de
contribuciones te dice exactamente qué mejorar para poder decidir. Probalo en
el ejercicio 2.

### 5. Lo que promediar NO arregla

Todo lo anterior supone que los errores son **aleatorios**. Si hay un error
**sistemático** —el calibre no cierra en cero, el cronómetro atrasa, siempre
soltás la regla un poco tarde— promediar no lo corrige: lo **enmascara**,
porque baja la dispersión y te da una falsa sensación de precisión.

Volvamos a los tiempos de reacción de la Clase 2. Supongamos que el método
de la regla en caída libre tiene un sesgo de +15 ms porque el que suelta
avisa sin querer.

In [ ]:
generador = np.random.default_rng(7)

verdadero = 0.230
sesgo = 0.015

mediciones = generador.normal(verdadero + sesgo, 0.030, size=500)
media, s, sem = lab.estadisticos(mediciones, verbose=False)

print(f"valor verdadero          : {verdadero:.4f} s")
print(f"resultado con 500 medidas: {lab.formatear(media, sem, 's')}")
print(f"discrepancia             : {abs(media - verdadero)/sem:.1f} sigma")
print()
print("Con 500 mediciones el SEM es chiquísimo y el resultado está")
print("escandalosamente mal. Precisión alta, exactitud baja.")

**Un ejemplo de investigación real, para que no parezca un problema de
juguete.** En las curvas corriente-tensión de una juntura memristiva, el
barrido de ida y el de vuelta no coinciden: hay histéresis. Promediar ida y
vuelta da un resultado con dispersión chica —muy reproducible— y
sistemáticamente corrido, porque el promedio de dos ramas físicamente
distintas no es ninguna de las dos. El mismo error conceptual, con
consecuencias caras.

Un error sistemático **no se arregla midiendo más veces**. Se arregla
encontrándolo: calibrando contra un patrón, cambiando el método, o midiendo
la misma cosa de dos maneras independientes y comparando (que es lo que
hicimos en la Clase 3).

### 6. Ejercicios

1. Propagá con sympy la incerteza del **volumen** de tu cilindro y comparala
   con la respuesta que escribiste en el ejercicio 3 del Colab 01.
2. Rehacé todo suponiendo que mediste el diámetro con **regla**
   ($\sigma_D = 0{,}5$ mm). ¿Cuántos materiales quedan compatibles? Y al
   revés, con **micrómetro** ($\sigma_D = 0{,}01$ mm): ¿quién domina ahora la
   tabla de contribuciones?
3. Para $\rho = 4m/(\pi D^2 h)$, ¿cuánto tendrías que mejorar la medición
   dominante para que la densidad salga con 0,3 % de error? ¿Es alcanzable
   con el equipamiento de la mesada?
4. Buscá en tus datos de tiempo de reacción de la Clase 2 evidencia de un
   sistemático: compará la primera mitad con la segunda, y compará el método
   de la regla con el del cronómetro si hiciste los dos.

In [ ]:
# Espacio de trabajo para los ejercicios.

### Entrega corta 2

Propagación completa de un caso propio, con la tabla de contribuciones y una
frase sobre qué medición mejorarías primero y por qué.